# Data Preprocessing and Transformation

## 0. Setup and configuration

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

# ----------------------------- CONFIGURATION -----------------------------
DATA_PATH = "../data/raw/dataset_eda.csv"
SAVE_PATH = "../data/processed/dataset_preprocessed.csv"
Z_THRESHOLD = 3.0                
OUTLIER_COLS = ["duration_ms", "loudness", "speechiness"]
COLS_TO_DROP = ["Unnamed: 0", "track_id", "artists", "album_name", "track_name"]
CATEGORICAL_COL = "track_genre"
# --------------------------------------------------------------------------

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head())

Loaded dataset: 114,000 rows x 21 columns


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## 1. Outlier removal (Z-score)

Z-scores are computed **once**, on the data as it is before removing anything. Recomputing them after dropping rows would change the mean and standard deviation and flag new outliers on every pass.

A row is removed if it is an outlier in **at least one** of the selected variables.

In [2]:
# Z-scores of the selected variables
z_scores = (df[OUTLIER_COLS] - df[OUTLIER_COLS].mean()) / df[OUTLIER_COLS].std()

# Boolean mask: True where |z| exceeds the threshold, per variable
is_outlier = z_scores.abs() > Z_THRESHOLD

# Outliers detected per variable (a row can be an outlier in more than one)
print("Outliers per variable:")
print(is_outlier.sum().to_string())

# Remove rows that are outliers in AT LEAST ONE variable
rows_to_remove = is_outlier.any(axis=1)

rows_before = len(df)
df = df.loc[~rows_to_remove].reset_index(drop=True)

print(f"\nRows before: {rows_before:,} | removed: {rows_to_remove.sum():,} "
      f"({rows_to_remove.mean():.2%}) | after: {len(df):,}")

Outliers per variable:
duration_ms     946
loudness       2475
speechiness    2079

Rows before: 114,000 | removed: 5,354 (4.70%) | after: 108,646


In [3]:
# Check: statistics of the cleaned variables
display(df[OUTLIER_COLS].describe().T)

,count,mean,std,min,25%,50%,75%,max
duration_ms,108646.0,224424.358715,76790.558952,0.000,175466.0000,213306.0000,260463.7500,549809.000
loudness,108646.0,-7.759086,4.000261,-23.346,-9.6580,-6.8600,-4.9320,4.532
speechiness,108646.0,0.074674,0.066561,0.000,0.0356,0.0485,0.0819,0.401


## 2. Dropping unnecessary columns

In [4]:
cols_present = [c for c in COLS_TO_DROP if c in df.columns]

# errors="ignore" avoids a crash if any of these columns was already removed
df = df.drop(columns=COLS_TO_DROP, errors="ignore")

print(f"Dropped columns: {cols_present}")
print(f"Shape: {df.shape}")

Dropped columns: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name']
Shape: (108646, 16)


## 3. One-hot encoding of `track_genre`

In [5]:
n_genres = df[CATEGORICAL_COL].nunique()

# One binary column per genre (dtype=int gives 0/1 instead of True/False)
df = pd.get_dummies(df, columns=[CATEGORICAL_COL], prefix="genre", dtype=int)

print(f"Genres encoded: {n_genres} | new shape: {df.shape}")
display(df.head())

Genres encoded: 114 | new shape: (108646, 129)


,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,genre_acoustic,genre_afrobeat,genre_alt-rock,genre_alternative,genre_ambient,genre_anime,genre_black-metal,genre_bluegrass,genre_blues,genre_brazil,...,genre_reggaeton,genre_rock,genre_rock-n-roll,genre_rockabilly,genre_romance,genre_sad,genre_salsa,genre_samba,genre_sertanejo,genre_show-tunes,genre_singer-songwriter,genre_ska,genre_sleep,genre_songwriter,genre_soul,genre_spanish,genre_study,genre_swedish,genre_synth-pop,genre_tango,genre_techno,genre_trance,genre_trip-hop,genre_turkish,genre_world-music
0,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 4. Final check

In [6]:
print(f"Final shape: {df.shape}")
print(f"Missing values: {int(df.isna().sum().sum())}")
print(f"Non-numeric columns left: {df.select_dtypes(exclude=np.number).columns.tolist() or 'none'}")

Final shape: (108646, 129)
Missing values: 0
Non-numeric columns left: ['explicit']


In [7]:
df.to_csv(SAVE_PATH, index=False)